# Notebook 10: Full Model Fine-Tuning for Classification

## 📚 Historical Context

**Timeline**: 2018-2020 - The Golden Age of Fine-Tuning

**Before Transfer Learning (Pre-2018)**:
- **Train from scratch**: Initialize random weights, train on task-specific data
- **Limited data**: Most NLP datasets had <100K examples
- **Poor generalization**: Models struggled on out-of-domain examples
- **Word2Vec/GloVe**: Only embeddings were pre-trained, rest trained from scratch
- **Task-specific architectures**: Different model for each task

**The Breakthrough (2018)**:
- **ULMFiT** (May 2018): First successful language model fine-tuning
- **ELMo** (Feb 2018): Contextual embeddings, but shallow integration
- **BERT** (Oct 2018): Pre-train on massive text, fine-tune on downstream tasks
  - Pre-trained on BooksCorpus (800M words) + Wikipedia (2.5B words)
  - Fine-tune entire model on GLUE tasks → State-of-the-art
  - 11/11 GLUE tasks improved dramatically

**Impact**:
- **2018-2020**: "BERT for everything" era
- **RoBERTa** (2019): Better pre-training → better fine-tuning
- **DistilBERT** (2019): 97% performance, 40% smaller, 60% faster
- **ALBERT** (2019): Parameter sharing for efficiency
- **DeBERTa** (2020): Disentangled attention → best GLUE scores

**Why Full Fine-Tuning Dominated (2018-2020)**:
1. Models were small enough (110M-340M params) to fit on single GPU
2. Memory: 12-16GB GPUs could handle full fine-tuning
3. Speed: Fine-tuning took hours, not days
4. Performance: Full fine-tuning gave best accuracy

**Modern Context (2023+)**:
- **Models too large**: 7B-70B params → can't fine-tune all parameters
- **PEFT methods**: LoRA, QLoRA, prefix tuning → Stage 2 of this course
- **But still important**: Small models (BERT, RoBERTa) still widely used in production
- **Edge deployment**: Smaller fully fine-tuned models > large LoRA models

## 🎯 What You'll Learn

1. **Loading Pre-trained Models** - Using transformers library
2. **Adding Classification Head** - Adapting encoder for task
3. **Forward and Backward Pass** - Understanding training mechanics
4. **Training Loop** - Implementing from scratch
5. **Evaluation Metrics** - Accuracy, F1, precision, recall
6. **Confusion Matrix** - Understanding errors
7. **Saving and Loading** - Model persistence

## 💡 Why This Matters

**When to use full fine-tuning**:
- Small models (< 1B parameters)
- High-stakes tasks requiring maximum accuracy
- Domain shift (medical, legal, technical text)
- Production deployment with latency constraints

**Real-world applications**:
- Content moderation (toxicity, spam detection)
- Customer support (ticket routing, sentiment)
- Healthcare (medical text classification)
- Finance (fraud detection, risk assessment)

---

In [ ]:
# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Part 1: Understanding Pre-trained Models

### What is a Pre-trained Model?

**Pre-training**:
- Train on massive unlabeled text (Wikipedia, books, web)
- Self-supervised objective (predict masked words, next token)
- Learn general language understanding
- Takes weeks, costs $100K+ in compute

**Fine-tuning**:
- Start from pre-trained weights
- Train on labeled task-specific data
- Adapt general knowledge to specific task
- Takes hours, costs $10-$100 in compute

### Popular Pre-trained Models:

| Model | Size | Pre-training | Best For |
|-------|------|--------------|----------|
| BERT-base | 110M | MLM on Wikipedia | General classification |
| RoBERTa-base | 125M | Better MLM training | Better than BERT |
| DistilBERT | 66M | Distilled from BERT | Speed + low memory |
| DeBERTa | 140M | Disentangled attention | State-of-the-art |
| ALBERT | 12M | Parameter sharing | Extremely efficient |

### Key Concepts:

**1. Tokenizer**
- Converts text to token IDs
- Must use SAME tokenizer as pre-training
- Includes special tokens: [CLS], [SEP], [PAD]

**2. Base Model**
- Transformer encoder layers
- Outputs hidden states for each token
- Shape: [batch_size, seq_len, hidden_dim]

**3. Classification Head**
- Added on top of base model
- Typically: [CLS] token → linear layer → classes
- This is what we fine-tune

In [ ]:
# Load pre-trained model and tokenizer
print("=" * 60)
print("LOADING PRE-TRAINED MODEL")
print("=" * 60)

model_name = "distilbert-base-uncased"  # Using DistilBERT for speed
# Alternatives:
# - "bert-base-uncased" (larger, slower, slightly better)
# - "roberta-base" (best quality, slower)
# - "microsoft/deberta-v3-base" (state-of-the-art)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
print(f"\nTokenizer: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")
print(f"Max length: {tokenizer.model_max_length}")

# Load configuration
config = AutoConfig.from_pretrained(model_name)
print(f"\nModel Configuration:")
print(f"  Hidden size: {config.hidden_size}")
print(f"  Num layers: {config.num_hidden_layers}")
print(f"  Num attention heads: {config.num_attention_heads}")
print(f"  Intermediate size: {config.intermediate_size}")

# Load base model (without classification head)
base_model = AutoModel.from_pretrained(model_name)

# Count parameters
total_params = sum(p.numel() for p in base_model.parameters())
trainable_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)

print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Memory (FP32): {total_params * 4 / 1e9:.2f} GB")
print(f"  Memory (FP16): {total_params * 2 / 1e9:.2f} GB")

print("\n" + "=" * 60)

## Part 2: Creating a Classification Model

### Architecture:

```
Input Text: "This movie is great!"
     ↓
Tokenizer: [CLS] this movie is great ! [SEP]
     ↓
Token IDs: [101, 2023, 3185, 2003, 2307, 999, 102]
     ↓
Embeddings: [seq_len, hidden_dim]
     ↓
Transformer Encoder (12 layers)
     ↓
Hidden States: [seq_len, hidden_dim]
     ↓
Extract [CLS] token (first token)
     ↓
Classification Head: Linear(hidden_dim, num_classes)
     ↓
Logits: [num_classes]
     ↓
Softmax → Probabilities
     ↓
Prediction: Positive (95% confidence)
```

### Why [CLS] Token?

**Historical reason (BERT paper)**:
- [CLS] = classification token
- Added at start of every sequence
- Through self-attention, aggregates information from all tokens
- Used as sequence representation

**Alternatives**:
- Mean pooling: Average all token embeddings
- Max pooling: Take max across sequence
- Last token: Used in GPT-style models

**Modern research**: Mean pooling often works as well or better, but [CLS] is convention

In [ ]:
class TextClassifier(nn.Module):
    """
    Text classification model with pre-trained transformer encoder.
    
    This is the standard architecture for fine-tuning:
    1. Pre-trained encoder (BERT, RoBERTa, etc.)
    2. Classification head (linear layer)
    3. Optional: Dropout for regularization
    """
    
    def __init__(self, model_name, num_classes, dropout=0.1, pooling='cls'):
        """
        Args:
            model_name: HuggingFace model name (e.g., 'bert-base-uncased')
            num_classes: Number of output classes
            dropout: Dropout probability before classification head
            pooling: How to pool token embeddings ('cls', 'mean', 'max')
        """
        super().__init__()
        
        self.pooling = pooling
        
        # Load pre-trained encoder
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        # Dropout for regularization
        # Applied before classification to prevent overfitting
        self.dropout = nn.Dropout(dropout)
        
        # Classification head
        # Simple linear layer: hidden_size → num_classes
        self.classifier = nn.Linear(hidden_size, num_classes)
        
        # Initialize classification head with small random weights
        # Important: Use small initialization to not disrupt pre-trained features
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)
    
    def forward(self, input_ids, attention_mask=None, return_hidden=False):
        """
        Forward pass.
        
        Args:
            input_ids: [batch_size, seq_len] token IDs
            attention_mask: [batch_size, seq_len] mask (1 for real tokens, 0 for padding)
            return_hidden: If True, return hidden states for visualization
        
        Returns:
            logits: [batch_size, num_classes] unnormalized class scores
        """
        # Pass through encoder
        # outputs.last_hidden_state: [batch_size, seq_len, hidden_size]
        # outputs.pooler_output: [batch_size, hidden_size] (only for some models)
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=return_hidden
        )
        
        # Get sequence representation
        if self.pooling == 'cls':
            # Use [CLS] token (first token)
            pooled = outputs.last_hidden_state[:, 0, :]  # [batch_size, hidden_size]
        
        elif self.pooling == 'mean':
            # Mean pooling over all tokens (excluding padding)
            hidden = outputs.last_hidden_state  # [batch_size, seq_len, hidden_size]
            
            # Expand attention mask for broadcasting
            # [batch_size, seq_len, 1]
            mask_expanded = attention_mask.unsqueeze(-1).float()
            
            # Sum embeddings (excluding padding)
            sum_embeddings = torch.sum(hidden * mask_expanded, dim=1)
            
            # Divide by number of non-padding tokens
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            pooled = sum_embeddings / sum_mask
        
        elif self.pooling == 'max':
            # Max pooling over all tokens
            hidden = outputs.last_hidden_state
            
            # Set padding tokens to very negative values
            mask_expanded = attention_mask.unsqueeze(-1).float()
            hidden = hidden.masked_fill(mask_expanded == 0, -1e9)
            
            # Max over sequence dimension
            pooled = torch.max(hidden, dim=1)[0]
        
        # Apply dropout
        pooled = self.dropout(pooled)
        
        # Classification head
        logits = self.classifier(pooled)  # [batch_size, num_classes]
        
        if return_hidden:
            return logits, outputs.hidden_states
        
        return logits
    
    def freeze_encoder(self, freeze=True):
        """
        Freeze encoder parameters.
        
        Useful for:
        - Training only classification head (faster, less memory)
        - Feature extraction
        - When you have very little training data
        """
        for param in self.encoder.parameters():
            param.requires_grad = not freeze
    
    def unfreeze_last_n_layers(self, n):
        """
        Unfreeze last n encoder layers.
        
        Useful for:
        - Gradual fine-tuning
        - Limited compute resources
        - Domain adaptation
        """
        # First freeze everything
        self.freeze_encoder(freeze=True)
        
        # Then unfreeze last n layers
        # Note: Specific to each model architecture
        # This is for BERT/DistilBERT/RoBERTa
        if hasattr(self.encoder, 'encoder'):
            layers = self.encoder.encoder.layer
        elif hasattr(self.encoder, 'transformer'):
            layers = self.encoder.transformer.layer
        else:
            raise ValueError("Unknown model architecture")
        
        # Unfreeze last n layers
        for layer in layers[-n:]:
            for param in layer.parameters():
                param.requires_grad = True

# Test the model
print("=" * 60)
print("CREATING CLASSIFICATION MODEL")
print("=" * 60)

# Create model for binary classification (e.g., sentiment: positive/negative)
num_classes = 2
model = TextClassifier(model_name, num_classes, dropout=0.1, pooling='cls')
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
encoder_params = sum(p.numel() for p in model.encoder.parameters())
classifier_params = sum(p.numel() for p in model.classifier.parameters())

print(f"\nModel Architecture:")
print(f"  Encoder: {model_name}")
print(f"  Num classes: {num_classes}")
print(f"  Pooling: CLS token")

print(f"\nParameters:")
print(f"  Encoder: {encoder_params:,} ({encoder_params/total_params*100:.1f}%)")
print(f"  Classifier: {classifier_params:,} ({classifier_params/total_params*100:.1f}%)")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Test forward pass
test_texts = [
    "This movie is absolutely fantastic!",
    "Terrible film, waste of time."
]

# Tokenize
encoded = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# Move to device
input_ids = encoded['input_ids'].to(device)
attention_mask = encoded['attention_mask'].to(device)

# Forward pass
with torch.no_grad():
    logits = model(input_ids, attention_mask)
    probs = F.softmax(logits, dim=-1)

print(f"\nTest Forward Pass:")
print(f"  Input shape: {input_ids.shape}")
print(f"  Logits shape: {logits.shape}")
print(f"  Predictions:")
for i, text in enumerate(test_texts):
    pred_class = torch.argmax(probs[i]).item()
    confidence = probs[i][pred_class].item()
    print(f"    Text: {text}")
    print(f"    Prediction: Class {pred_class} ({confidence:.2%} confidence)")
    print(f"    Probabilities: {probs[i].cpu().numpy()}")

print("\n" + "=" * 60)

## Part 3: Creating a Dataset

### Dataset Requirements:

1. **Text**: Raw input strings
2. **Labels**: Integer class indices (0, 1, 2, ...)
3. **Tokenization**: Convert text to token IDs
4. **Batching**: Group examples with padding

### Best Practices:

- **Tokenize on-the-fly**: Don't pre-tokenize entire dataset (memory)
- **Dynamic padding**: Pad to longest sequence in batch, not max_length
- **Data cleaning**: Remove special characters, normalize text
- **Stratified splits**: Maintain class balance in train/val/test

### Real Dataset Example: IMDb Sentiment

We'll create a synthetic sentiment dataset for demonstration.

In [ ]:
class SentimentDataset(Dataset):
    """
    Custom dataset for text classification.
    
    This is a template you can adapt for any classification task.
    """
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        """
        Args:
            texts: List of input strings
            labels: List of integer labels
            tokenizer: HuggingFace tokenizer
            max_length: Maximum sequence length
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        """
        Get a single example.
        
        Returns:
            dict with keys: input_ids, attention_mask, labels
        """
        text = self.texts[idx]
        label = self.labels[idx]
        
        # Tokenize text
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding='max_length',  # Pad to max_length
            truncation=True,       # Truncate if longer
            return_tensors='pt'    # Return PyTorch tensors
        )
        
        # Remove batch dimension (added by return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(0),        # [seq_len]
            'attention_mask': encoding['attention_mask'].squeeze(0),  # [seq_len]
            'labels': torch.tensor(label, dtype=torch.long)       # scalar
        }

# Create synthetic sentiment dataset
# In practice, you'd load from file or API
print("=" * 60)
print("CREATING DATASET")
print("=" * 60)

# Synthetic examples
# Class 0: Negative sentiment
# Class 1: Positive sentiment
train_texts = [
    # Positive examples
    "This movie is absolutely fantastic! Best film I've seen all year.",
    "Amazing performance by the lead actor. Highly recommended!",
    "Brilliant script and excellent cinematography. A masterpiece.",
    "Loved every minute of it. Will definitely watch again.",
    "Outstanding film with a powerful message. Must-see!",
    
    # Negative examples
    "Terrible movie. Complete waste of time and money.",
    "The worst film I've ever seen. Avoid at all costs.",
    "Boring plot and terrible acting. Very disappointed.",
    "Couldn't even finish watching it. Absolutely awful.",
    "Save your money and skip this disaster of a film.",
] * 50  # Duplicate to create larger dataset

train_labels = [1, 1, 1, 1, 1, 0, 0, 0, 0, 0] * 50

# Create validation set
val_texts = [
    "Great movie with an interesting storyline.",
    "Not worth watching. Very poor quality.",
    "Excellent acting and direction. Loved it!",
    "Boring and predictable. Waste of time.",
] * 10

val_labels = [1, 0, 1, 0] * 10

print(f"Training examples: {len(train_texts)}")
print(f"Validation examples: {len(val_texts)}")

# Class distribution
print(f"\nClass distribution (train):")
print(f"  Positive: {sum(1 for l in train_labels if l == 1)} ({sum(1 for l in train_labels if l == 1)/len(train_labels)*100:.1f}%)")
print(f"  Negative: {sum(1 for l in train_labels if l == 0)} ({sum(1 for l in train_labels if l == 0)/len(train_labels)*100:.1f}%)")

# Create datasets
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_length=128)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, max_length=128)

# Create dataloaders
batch_size = 16
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # Shuffle training data
    num_workers=0  # Set to > 0 for faster loading on multi-core systems
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False  # Don't shuffle validation data
)

print(f"\nDataLoader Configuration:")
print(f"  Batch size: {batch_size}")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")

# Test dataloader
batch = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  input_ids shape: {batch['input_ids'].shape}")
print(f"  attention_mask shape: {batch['attention_mask'].shape}")
print(f"  labels shape: {batch['labels'].shape}")

print("\n" + "=" * 60)

## Part 4: Training Loop Implementation

### Training Loop Components:

1. **Forward Pass**: Input → Model → Logits
2. **Loss Calculation**: CrossEntropyLoss(logits, labels)
3. **Backward Pass**: Loss → Gradients
4. **Optimizer Step**: Update weights
5. **Learning Rate Schedule**: Adjust LR during training

### Key Hyperparameters:

**Learning Rate**:
- Too high: Unstable training, loss spikes
- Too low: Slow convergence, might not converge
- Typical range: 1e-5 to 5e-5 for fine-tuning
- Larger models need smaller LR

**Learning Rate Schedule**:
- **Warmup**: Gradually increase LR from 0 to max_lr
  - Prevents early instability
  - Typical: 5-10% of total steps
- **Linear decay**: Decrease LR to 0 over training
  - Better final performance
  - Standard in transformer fine-tuning

**Number of Epochs**:
- Small datasets: 3-10 epochs
- Large datasets: 1-3 epochs
- Monitor validation loss to detect overfitting

**Batch Size**:
- Limited by GPU memory
- Typical: 8-32 for fine-tuning
- Larger batch = more stable, but might need to adjust LR

### Optimizer Choice:

**AdamW** (default for transformers):
- Adam with weight decay fix
- Better regularization than Adam
- Default betas: (0.9, 0.999)
- Default weight decay: 0.01

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, device):
    """
    Train for one epoch.
    
    Args:
        model: Classification model
        dataloader: Training dataloader
        optimizer: AdamW optimizer
        scheduler: Learning rate scheduler
        device: 'cuda' or 'cpu'
    
    Returns:
        avg_loss: Average loss over epoch
        accuracy: Training accuracy
    """
    model.train()  # Set to training mode (enables dropout, etc.)
    
    total_loss = 0
    correct_predictions = 0
    total_predictions = 0
    
    # Progress bar
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch in progress_bar:
        # Move batch to device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Zero gradients from previous step
        # Important: Gradients accumulate by default in PyTorch
        optimizer.zero_grad()
        
        # Forward pass
        logits = model(input_ids, attention_mask)
        
        # Calculate loss
        # CrossEntropyLoss combines softmax + negative log-likelihood
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        # Compute gradients of loss w.r.t. all parameters
        loss.backward()
        
        # Clip gradients to prevent exploding gradients
        # Typical value: 1.0
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update parameters
        optimizer.step()
        
        # Update learning rate
        scheduler.step()
        
        # Calculate accuracy
        predictions = torch.argmax(logits, dim=-1)
        correct_predictions += (predictions == labels).sum().item()
        total_predictions += labels.size(0)
        
        # Accumulate loss
        total_loss += loss.item()
        
        # Update progress bar
        progress_bar.set_postfix({
            'loss': loss.item(),
            'lr': scheduler.get_last_lr()[0]
        })
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct_predictions / total_predictions
    
    return avg_loss, accuracy


def evaluate(model, dataloader, device):
    """
    Evaluate model on validation/test set.
    
    Args:
        model: Classification model
        dataloader: Validation dataloader
        device: 'cuda' or 'cpu'
    
    Returns:
        avg_loss: Average loss
        accuracy: Accuracy
        all_predictions: List of predictions
        all_labels: List of true labels
    """
    model.eval()  # Set to evaluation mode (disables dropout, etc.)
    
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    # No gradients needed for evaluation
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Evaluating")
        
        for batch in progress_bar:
            # Move batch to device
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            # Forward pass
            logits = model(input_ids, attention_mask)
            
            # Calculate loss
            loss = F.cross_entropy(logits, labels)
            total_loss += loss.item()
            
            # Get predictions
            predictions = torch.argmax(logits, dim=-1)
            
            # Store predictions and labels
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(all_labels, all_predictions)
    
    return avg_loss, accuracy, all_predictions, all_labels


# Initialize training components
print("=" * 60)
print("INITIALIZING TRAINING")
print("=" * 60)

# Hyperparameters
num_epochs = 3
learning_rate = 2e-5
weight_decay = 0.01
warmup_steps = int(0.1 * len(train_loader) * num_epochs)  # 10% of total steps

print(f"\nHyperparameters:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Batch size: {batch_size}")
print(f"  Weight decay: {weight_decay}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Total training steps: {len(train_loader) * num_epochs}")

# Optimizer
# AdamW: Adam with decoupled weight decay
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay
)

# Learning rate scheduler
# Warmup + linear decay to 0
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("\nOptimizer: AdamW")
print("Scheduler: Linear warmup + decay")
print("\n" + "=" * 60)

In [ ]:
# Training loop
print("=" * 60)
print("TRAINING MODEL")
print("=" * 60)

# Track metrics
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

best_val_loss = float('inf')
best_model_state = None

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 60)
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device
    )
    
    # Evaluate
    val_loss, val_acc, _, _ = evaluate(model, val_loader, device)
    
    # Store metrics
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    # Print metrics
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        print(f"  ✓ New best model! (Val Loss: {val_loss:.4f})")

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"Best validation loss: {best_val_loss:.4f}")

# Load best model
model.load_state_dict(best_model_state)
print("Loaded best model weights")

## Part 5: Visualization and Analysis

### Learning Curves:
- **Training loss decreasing**: Model is learning
- **Validation loss decreasing**: Generalizing well
- **Validation loss increasing**: Overfitting
- **Both losses plateau**: Reached optimal performance

### Signs of Overfitting:
- Train accuracy >> Val accuracy
- Train loss << Val loss
- Val loss increases while train loss decreases

### Solutions:
- Increase dropout
- Increase weight decay
- Early stopping
- Data augmentation
- Use smaller model

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, num_epochs + 1)

# Loss curves
axes[0].plot(epochs_range, train_losses, 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, val_losses, 'r-', label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(epochs_range, train_accuracies, 'b-', label='Train Acc', linewidth=2)
axes[1].plot(epochs_range, val_accuracies, 'r-', label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Learning Curve Analysis:")
print(f"  Final train loss: {train_losses[-1]:.4f}")
print(f"  Final val loss: {val_losses[-1]:.4f}")
print(f"  Train-val gap: {abs(train_losses[-1] - val_losses[-1]):.4f}")
print(f"  Best val loss: {best_val_loss:.4f} (Epoch {np.argmin(val_losses) + 1})")

if val_losses[-1] > val_losses[0]:
    print("  ⚠️ Warning: Validation loss increased - possible overfitting")
elif abs(train_losses[-1] - val_losses[-1]) > 0.5:
    print("  ⚠️ Warning: Large train-val gap - overfitting detected")
else:
    print("  ✓ Training looks healthy")

## Part 6: Comprehensive Evaluation

### Beyond Accuracy:

**Precision**:
- Of all predicted positives, how many are actually positive?
- Formula: TP / (TP + FP)
- Important when false positives are costly

**Recall** (Sensitivity):
- Of all actual positives, how many did we find?
- Formula: TP / (TP + FN)
- Important when false negatives are costly

**F1 Score**:
- Harmonic mean of precision and recall
- Formula: 2 * (Precision * Recall) / (Precision + Recall)
- Balances precision and recall

### When to Use Each:

- **Accuracy**: Balanced datasets, equal cost for all errors
- **Precision**: Spam detection (false positive = user annoyance)
- **Recall**: Disease detection (false negative = missed diagnosis)
- **F1**: Imbalanced datasets, both errors matter

In [ ]:
# Get predictions on validation set
print("=" * 60)
print("COMPREHENSIVE EVALUATION")
print("=" * 60)

val_loss, val_acc, predictions, true_labels = evaluate(model, val_loader, device)

# Calculate metrics
precision = precision_score(true_labels, predictions, average='binary')
recall = recall_score(true_labels, predictions, average='binary')
f1 = f1_score(true_labels, predictions, average='binary')

print(f"\nPerformance Metrics:")
print(f"  Accuracy:  {val_acc:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Detailed classification report
print(f"\nClassification Report:")
print(classification_report(
    true_labels,
    predictions,
    target_names=['Negative', 'Positive']
))

# Confusion matrix
cm = confusion_matrix(true_labels, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive']
)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')

# Add percentages
for i in range(2):
    for j in range(2):
        percentage = cm[i, j] / cm.sum() * 100
        plt.text(
            j, i + 0.3,
            f'({percentage:.1f}%)',
            ha='center',
            va='center',
            fontsize=10,
            color='gray'
        )

plt.tight_layout()
plt.show()

print(f"\nConfusion Matrix Interpretation:")
print(f"  True Negatives:  {cm[0, 0]} (correctly predicted negative)")
print(f"  False Positives: {cm[0, 1]} (predicted positive, actually negative)")
print(f"  False Negatives: {cm[1, 0]} (predicted negative, actually positive)")
print(f"  True Positives:  {cm[1, 1]} (correctly predicted positive)")

print("\n" + "=" * 60)

## Part 7: Testing on New Examples

### Real-world Deployment:

1. **Preprocessing**: Same as training (tokenization, padding)
2. **Inference**: Forward pass without gradients
3. **Post-processing**: Convert logits to predictions
4. **Confidence**: Use softmax probabilities

### Production Considerations:

- **Batching**: Process multiple examples together for efficiency
- **Caching**: Cache tokenization for repeated queries
- **Quantization**: Convert to INT8 for faster inference
- **ONNX**: Export to ONNX for deployment
- **Serving**: Use FastAPI, TorchServe, or TensorFlow Serving

In [ ]:
def predict_sentiment(model, tokenizer, texts, device, threshold=0.5):
    """
    Predict sentiment for new texts.
    
    Args:
        model: Trained classification model
        tokenizer: HuggingFace tokenizer
        texts: List of input strings
        device: 'cuda' or 'cpu'
        threshold: Confidence threshold for positive class
    
    Returns:
        predictions: List of (class, confidence, probs) tuples
    """
    model.eval()
    
    # Tokenize
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    
    # Predict
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs = F.softmax(logits, dim=-1)
    
    # Convert to predictions
    predictions = []
    for i in range(len(texts)):
        class_probs = probs[i].cpu().numpy()
        pred_class = np.argmax(class_probs)
        confidence = class_probs[pred_class]
        
        predictions.append({
            'class': pred_class,
            'label': 'Positive' if pred_class == 1 else 'Negative',
            'confidence': confidence,
            'probs': class_probs
        })
    
    return predictions


# Test on new examples
print("=" * 60)
print("TESTING ON NEW EXAMPLES")
print("=" * 60)

test_texts = [
    "This movie exceeded all my expectations. Truly brilliant!",
    "Absolutely terrible. Don't waste your money.",
    "It was okay, nothing special.",
    "One of the best films I've ever seen!",
    "Boring and uninspired. Very disappointing.",
    "The acting was superb and the story was captivating."
]

predictions = predict_sentiment(model, tokenizer, test_texts, device)

print("\nPredictions:\n")
for i, (text, pred) in enumerate(zip(test_texts, predictions), 1):
    print(f"{i}. Text: {text}")
    print(f"   Prediction: {pred['label']}")
    print(f"   Confidence: {pred['confidence']:.2%}")
    print(f"   Probabilities: Negative={pred['probs'][0]:.2%}, Positive={pred['probs'][1]:.2%}")
    print()

print("=" * 60)

## Part 8: Saving and Loading Models

### What to Save:

1. **Model Weights**: Trained parameters
2. **Tokenizer**: Vocabulary and special tokens
3. **Config**: Model architecture, hyperparameters
4. **Training Info**: Metrics, best epoch, etc.

### Save Formats:

**PyTorch (.pt, .pth)**:
- Native PyTorch format
- Fastest to load
- Use: `torch.save(model.state_dict(), path)`

**HuggingFace (directory)**:
- Standard for transformers
- Includes config and tokenizer
- Easy to share and deploy
- Use: `model.save_pretrained(path)`

**ONNX (.onnx)**:
- Framework-agnostic
- Faster inference
- Compatible with many deployment tools
- Use: `torch.onnx.export()`

In [ ]:
import os
import json

# Save model
print("=" * 60)
print("SAVING MODEL")
print("=" * 60)

save_dir = "./sentiment_classifier"
os.makedirs(save_dir, exist_ok=True)

# Method 1: Save PyTorch state dict (smallest file)
torch.save({
    'model_state_dict': model.state_dict(),
    'model_config': {
        'model_name': model_name,
        'num_classes': num_classes,
        'dropout': 0.1,
        'pooling': 'cls'
    },
    'training_info': {
        'best_val_loss': best_val_loss,
        'final_val_acc': val_accuracies[-1],
        'num_epochs': num_epochs,
        'learning_rate': learning_rate
    }
}, os.path.join(save_dir, 'model.pt'))

print(f"\nSaved PyTorch model to: {save_dir}/model.pt")

# Method 2: Save tokenizer (HuggingFace format)
tokenizer.save_pretrained(save_dir)
print(f"Saved tokenizer to: {save_dir}/")

# Save training metrics
metrics = {
    'train_losses': train_losses,
    'train_accuracies': train_accuracies,
    'val_losses': val_losses,
    'val_accuracies': val_accuracies,
    'best_val_loss': best_val_loss,
    'final_metrics': {
        'accuracy': val_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }
}

with open(os.path.join(save_dir, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to: {save_dir}/metrics.json")

# List saved files
print(f"\nSaved files:")
for file in os.listdir(save_dir):
    file_path = os.path.join(save_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"  {file}: {size_mb:.2f} MB")

print("\n" + "=" * 60)

# Load model (demonstration)
print("LOADING MODEL")
print("=" * 60)

# Load checkpoint
checkpoint = torch.load(os.path.join(save_dir, 'model.pt'))

# Recreate model
config = checkpoint['model_config']
loaded_model = TextClassifier(
    model_name=config['model_name'],
    num_classes=config['num_classes'],
    dropout=config['dropout'],
    pooling=config['pooling']
)

# Load weights
loaded_model.load_state_dict(checkpoint['model_state_dict'])
loaded_model = loaded_model.to(device)
loaded_model.eval()

# Load tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(save_dir)

print(f"\nLoaded model from: {save_dir}")
print(f"Training info:")
for key, value in checkpoint['training_info'].items():
    print(f"  {key}: {value}")

# Test loaded model
test_text = ["This is an excellent movie!"]
pred = predict_sentiment(loaded_model, loaded_tokenizer, test_text, device)
print(f"\nTest prediction: {pred[0]['label']} ({pred[0]['confidence']:.2%} confidence)")

print("\n" + "=" * 60)

## 📊 Summary

### What We Learned:

1. **Pre-trained Models**
   - Transfer learning from massive pre-training
   - BERT, RoBERTa, DistilBERT variants
   - Loading with HuggingFace transformers

2. **Classification Architecture**
   - Encoder + classification head
   - [CLS] token pooling vs mean/max pooling
   - Small classification head initialization

3. **Training Loop**
   - Forward → Loss → Backward → Optimizer step
   - AdamW optimizer with weight decay
   - Learning rate warmup + linear decay
   - Gradient clipping for stability

4. **Evaluation**
   - Multiple metrics: accuracy, precision, recall, F1
   - Confusion matrix visualization
   - Learning curves for diagnosing issues

5. **Deployment**
   - Saving models and tokenizers
   - Loading for inference
   - Batch prediction API

### Key Hyperparameters:

| Parameter | Typical Range | Impact |
|-----------|--------------|--------|
| Learning Rate | 1e-5 to 5e-5 | Higher = faster but less stable |
| Batch Size | 8 to 32 | Limited by GPU memory |
| Epochs | 2 to 5 | More = better, but watch for overfitting |
| Warmup | 5-10% steps | Stabilizes early training |
| Weight Decay | 0.01 to 0.1 | Regularization strength |
| Dropout | 0.1 to 0.3 | Higher = more regularization |

### Common Issues and Solutions:

**Loss not decreasing**:
- Learning rate too low → increase
- Learning rate too high → decrease
- Bad initialization → try different seed

**Overfitting**:
- Increase dropout
- Increase weight decay
- Use early stopping
- Get more data or augment

**Out of memory**:
- Reduce batch size
- Reduce max sequence length
- Use gradient accumulation (next notebook)
- Use mixed precision training (Notebook 15)

**Slow training**:
- Use smaller model (DistilBERT)
- Reduce sequence length
- Use mixed precision (FP16)
- Freeze encoder layers

### Performance Baselines:

**Binary Sentiment (IMDb)**:
- Random: 50% accuracy
- Logistic regression: ~88% accuracy
- BERT-base fine-tuned: ~94% accuracy
- RoBERTa-large fine-tuned: ~96% accuracy

**Multi-class (AG News)**:
- Random: 25% accuracy
- CNN baseline: ~87% accuracy
- BERT-base fine-tuned: ~94% accuracy
- RoBERTa-large fine-tuned: ~96% accuracy

### When to Use Full Fine-Tuning:

✅ **Use when**:
- Model is small (< 1B parameters)
- You have sufficient GPU memory
- You need maximum accuracy
- Domain shift is large (medical, legal)

❌ **Don't use when**:
- Model is large (> 7B parameters)
- Limited GPU memory
- Very small dataset (< 1000 examples)
- Fast iteration needed → use LoRA instead

### Historical Impact:

**2018: BERT Revolution**
- Pre-train + fine-tune paradigm
- 11/11 GLUE tasks state-of-the-art
- Replaced task-specific architectures

**2019-2020: Optimization Era**
- RoBERTa: Better pre-training recipe
- DistilBERT: Smaller, faster, cheaper
- ELECTRA: Better pre-training objective

**2021-2023: Scale Up**
- Models grew too large for full fine-tuning
- Rise of PEFT (LoRA, adapters, prompt tuning)
- Full fine-tuning still used for small models

### Next Notebook:
**11_full_finetuning_generation.ipynb**

We'll explore:
- Fine-tuning GPT-2 for text generation
- Causal language modeling objective
- Perplexity as evaluation metric
- Temperature and top-p sampling
- Comparing generation quality before/after
- Common generation issues (repetition, incoherence)

---

**Historical Note**: The "Pre-train, then fine-tune" paradigm dominated NLP from 2018-2022. It was revolutionary because:
1. **Democratized NLP**: Small teams could achieve SOTA with limited data
2. **Task-agnostic**: Same model for all tasks
3. **Sample efficient**: Needed 10x less labeled data
4. **Performance**: Massive improvements across all benchmarks

While modern LLMs use PEFT, understanding full fine-tuning is essential because:
- Edge deployment still uses small, fully fine-tuned models
- Foundation for understanding PEFT methods
- Many production systems still use BERT/RoBERTa
- Core concepts apply to all fine-tuning approaches